# Task 1: Your First Virtual Embryo Challenge Submission

The official [`veckit`](https://github.com/aristoteleo/veckit) tutorial shows you how to *score* a
prediction once you already have one, using tiny bundled 150-cell samples. It deliberately stops at
"swap `--input` for your own model's prediction." This notebook picks up there, for **Task 1**
(temporal scRNA-seq extrapolation): loading the real data, building an actual first prediction (a
floor baseline and one honest step above it), then packaging and locally sanity-checking a
submission before you spend a submission slot on it.

**What Task 1 asks:** predict the gene-expression distribution of mouse embryo cells at a
developmental stage you have not observed, given earlier stages. Train stages are E8.5 and E9.5; the
public validation leaderboard scores against E10.5; the hidden test ranking uses E12.5. Nothing is
observed between E10.5 and E12.5, so this is genuine extrapolation, not interpolation. Full task
definition: [virtualembryo.ai/challenge/tasks](https://virtualembryo.ai/challenge/tasks).

**This notebook contains no Challenge data.** Per the Challenge Terms of Use (clause 14), the
released datasets are unpublished and may not be redistributed. You must register at
[virtualembryo.ai](https://virtualembryo.ai) and download the files yourself. See the next cell.


## Prerequisites

1. Register at [virtualembryo.ai](https://virtualembryo.ai) and sign in.
2. Download `E8.5_RNA.h5ad` (571 MB) and `E9.5_RNA.h5ad` (590 MB) from
   [virtualembryo.ai/challenge/data](https://virtualembryo.ai/challenge/data).
3. Put both files in `../data/` next to this notebook (already gitignored, so they will never be
   committed). If you keep your copy in your own S3 bucket instead, set `DATA_DIR=s3://bucket/prefix`
   before starting Jupyter and the next cell copies them down with your own AWS credentials.
4. From the repo root: `uv sync`, then `uv run jupyter lab notebooks/getting-started.ipynb`.

Both files are AnnData `.h5ad` containers: 32,285 genes, log1p-normalized expression in `.X`,
cell-type annotations in `obs['celltype']`, no spatial coordinates for Task 1.


In [ ]:
import json
import os
import urllib.request
from pathlib import Path

import anndata as ad
import numpy as np
from scipy import sparse

FILES = ["E8.5_RNA.h5ad", "E9.5_RNA.h5ad"]
OUT = Path("../data")  # predictions/submissions land here (already gitignored)
OUT.mkdir(parents=True, exist_ok=True)

# Where the two training files are. A local folder (default), or s3://bucket/prefix if you keep
# your own downloaded copy in your own bucket. Nothing here fetches from the Challenge itself:
# its Terms of Use bar mirroring the data, so the copy has to be yours.
DATA_DIR = os.environ.get("DATA_DIR", str(OUT))

if DATA_DIR.startswith("s3://"):
    import boto3

    bucket, _, prefix = DATA_DIR[len("s3://"):].partition("/")
    s3 = boto3.client("s3")
    for name in FILES:
        dest = OUT / name
        if dest.exists():
            continue
        key = f"{prefix.rstrip('/')}/{name}".lstrip("/")
        print(f"copying s3://{bucket}/{key} -> {dest}")
        s3.download_file(bucket, key, str(dest))
    DATA = OUT
else:
    DATA = Path(DATA_DIR)

missing = [name for name in FILES if not (DATA / name).exists()]
assert not missing, f"missing {missing} in {DATA.resolve()}: download them from virtualembryo.ai first (see above)"


def fetch(url: str) -> bytes:
    # virtualembryo.ai 403s a bare urllib request with no User-Agent
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    return urllib.request.urlopen(req).read()


def dense(a: ad.AnnData) -> np.ndarray:
    return a.X.toarray() if sparse.issparse(a.X) else np.asarray(a.X)


In [ ]:
a85 = ad.read_h5ad(DATA / "E8.5_RNA.h5ad")
a95 = ad.read_h5ad(DATA / "E9.5_RNA.h5ad")

print("E8.5:", a85.shape)
print("E9.5:", a95.shape)
print("same gene panel, same order:", list(a85.var_names) == list(a95.var_names))
print("X range:", dense(a85).min(), "to", dense(a85).max(), "(log1p-normalized, so this is expected)")


## Why this is hard

This is not a small population drift. Compare cell-type composition between the two training
stages you actually have:


In [ ]:
import pandas as pd

comp85 = a85.obs["celltype"].value_counts(normalize=True).mul(100).round(2)
comp95 = a95.obs["celltype"].value_counts(normalize=True).mul(100).round(2)
shared = sorted(set(comp85.index) & set(comp95.index))

print(f"E8.5: {a85.n_obs} cells, {comp85.shape[0]} cell types")
print(f"E9.5: {a95.n_obs} cells, {comp95.shape[0]} cell types")
print(f"shared cell types: {len(shared)} -> {shared}")
print()
print("cell types in E9.5 that did not exist (by this label) at E8.5:")
print(sorted(set(comp95.index) - set(comp85.index)))


Roughly half the cell-type labels at E9.5 are not present at E8.5. A model that only nudges
existing cells around cannot account for that: some of what it needs to predict is cell-fate
change, not just expression drift within a fixed population. Keep this in mind: the baseline below
is deliberately simple and does **not** attempt to model this.


## Baseline 0: the floor, `copy_last`

The simplest possible prediction: reuse the most recent real observation verbatim, unchanged. This
is the official floor baseline (`"floor_model": "copy_last"` in the Task 1 panel spec). Every real
model should beat it.


In [ ]:
pred_floor = a85.copy()
pred_floor.write_h5ad(OUT / "pred_floor.h5ad")


## Baseline 1: pseudobulk mean-shift extrapolation

One honest step up: measure the average per-gene shift between the two stages you have (E9.5
pseudobulk mean minus E8.5 pseudobulk mean, in the same log1p space the data already ships in),
then apply that same shift to every cell. This captures *directional* change, which `copy_last`
cannot, while staying simple enough to fully understand and audit.

It will not invent new cell types (see the composition gap above), and translating every cell by
the same vector does not change the gene-gene covariance structure at all, so don't expect it to
help metrics that specifically reward correct covariance. Watch for both effects in the scores
below.


In [ ]:
X85 = dense(a85)
X95 = dense(a95)

delta = X95.mean(axis=0) - X85.mean(axis=0)  # observed E8.5 -> E9.5 shift, one gene at a time

X_shift = np.clip(X85 + delta, 0, None).astype(np.float32)  # metrics require nonnegative log-normalized data
pred_shift = a85.copy()
pred_shift.X = X_shift
pred_shift.write_h5ad(OUT / "pred_shift.h5ad")


## Scoring locally, honestly

The real E10.5 target is held out. You don't have it, and can't. So, exactly like `veckit`'s own
tutorial does, use a **pseudo target**: pretend E9.5 is the unknown stage, using the real
E8.5-to-E9.5 pair you do have ground truth for. This tells you whether your *mechanism* works, not
your real leaderboard score. The actual E9.5-to-E10.5 shift you'll extrapolate for the real
submission is a different, unknown one, so don't over-trust these exact numbers.


In [ ]:
from veckit import score

r_floor = score(task="T1", input=str(OUT / "pred_floor.h5ad"),
                 target=str(DATA / "E9.5_RNA.h5ad"), reference=str(DATA / "E8.5_RNA.h5ad"))
r_shift = score(task="T1", input=str(OUT / "pred_shift.h5ad"),
                 target=str(DATA / "E9.5_RNA.h5ad"), reference=str(DATA / "E8.5_RNA.h5ad"))

primary = ["de_score", "de_direction", "mmd_u", "variogram"]  # the metrics T1:val's official anchors normalize
print(f"{'metric':<14}{'floor (copy_last)':<20}{'mean-shift':<12}")
for k in primary:
    print(f"{k:<14}{r_floor['metrics'][k]:<20}{r_shift['metrics'][k]:<12}")


`de_score` and `de_direction` are 0 for the floor by construction. `copy_last` predicts no change
at all, so there is no direction to get right or wrong. On a 150-cell toy calibration run using
`veckit`'s own public sample files (not the real Challenge data, just to sanity-check this exact
code before you point it at your real files), mean-shift took `de_score` from 0.0 to 0.58 and
`de_direction` from 0.0 to 0.998, while `variogram` got slightly worse, consistent with the
mechanism above: it fixes direction, it does nothing for covariance. Your real numbers will differ;
what matters is the same pattern of trade-offs, not these exact figures.

The rest of what `score()` returns (`energy_distance`, `pb_rel_err`, `library_size_ratio`,
`variance_ratio`, `composition_JSD`, `pseudobulk_pearson`) are useful diagnostics but aren't part of
the four metrics the official T1:val anchors normalize. See
[virtualembryo.ai/challenge/evaluation](https://virtualembryo.ai/challenge/evaluation).


## Building the real submission: E9.5 to E10.5

Apply the *same* learned mechanism one stage forward, where you don't have ground truth: shift the
real E9.5 cells by the same kind of delta, this time measured off the E8.5-to-E9.5 pair, to predict
E10.5. Then shape the result to match the live Task 1 validation panel spec: gene order and a cell
count the scorer will accept.


In [ ]:
spec = json.loads(fetch("https://virtualembryo.ai/challenge/panels/index.json"))["T1:val"]
panel_genes = fetch(
    "https://virtualembryo.ai/challenge/panels/" + spec["genes_file"]
).decode().splitlines()

print(spec["label"], "-", spec["n_genes"], "genes, cells in [", spec["min_cells"], ",", spec["max_cells"], "]")
print("panel matches your files' gene order already:", panel_genes == list(a95.var_names))


In [ ]:
X_next = np.clip(X95 + delta, 0, None).astype(np.float32)

# E9.5 has ~17k cells; the panel wants between min_cells and max_cells. Subsample down if needed.
n_target = min(spec["max_cells"], max(spec["min_cells"], a95.n_obs))
rng = np.random.default_rng(0)
idx = rng.choice(a95.n_obs, size=n_target, replace=False) if a95.n_obs > n_target else np.arange(a95.n_obs)

submission = ad.AnnData(X=X_next[idx], var=a95.var.copy(), obs=a95.obs.iloc[idx].copy())
if panel_genes != list(submission.var_names):
    submission = submission[:, panel_genes].copy()  # only triggers if the panel ever reorders; see README

submission.write_h5ad(OUT / "submission_T1_val.h5ad")
print("wrote", OUT / "submission_T1_val.h5ad", submission.shape)


## Format sanity check

Mirrors the same checks `veckit`'s own `score_h5ad.py` runs on every file it loads: gene panel
order, finite nonnegative values, cell count in range. This cannot tell you how good the prediction
is (no ground truth exists for E10.5 outside the Challenge's own servers). It only confirms the file
is shaped correctly.

(If you're dry-running this notebook against `veckit`'s tiny 150-cell public samples instead of your
real downloaded files, the cell-count assertion below will fail: 150 is below `min_cells`. That's
expected. Those samples exist to exercise the scorer, not to produce a submission-shaped file. It
will pass on your real ~17k-cell E9.5 file.)


In [ ]:
check = ad.read_h5ad(OUT / "submission_T1_val.h5ad")
Xc = dense(check)

assert list(check.var_names) == panel_genes, "gene panel / order mismatch"
assert np.isfinite(Xc).all(), "non-finite values present"
assert (Xc >= 0).all(), "negative values present -- metrics expect log-normalized nonnegative data"
assert spec["min_cells"] <= check.n_obs <= spec["max_cells"], f"cell count {check.n_obs} outside allowed range"

print("all format checks passed:", check.shape)


## Submitting

Sign in at [virtualembryo.ai/challenge/account/submissions](https://virtualembryo.ai/challenge/account/submissions)
and upload `submission_T1_val.h5ad`. Check
[virtualembryo.ai/challenge/rules](https://virtualembryo.ai/challenge/rules) for the current
submission cadence limit before you use one.

## Where to go from here

This baseline moves the whole population by one shared vector. It cannot invent the cell types
missing from the composition gap above, and it ignores gene-gene covariance entirely. Ideas that go
further, roughly in order of effort:

- **Per-cell-type shift** instead of one global shift: compute the delta separately for each of
  the shared cell types, and handle emerging types some other way (e.g. via the frozen probe
  `veckit` already trains on the target's cell types).
- **Nearest-neighbor / optimal-transport extrapolation** in PCA space instead of a linear shift in
  gene space.
- The organisers' own reference baselines:
  [virtualembryo.ai/challenge/baselines](https://virtualembryo.ai/challenge/baselines).

## Acknowledgments

Data, task design, and the `veckit` scorer are the work of Dr. Neil Chi's group and the Qiu Lab
([virtualembryo.ai](https://virtualembryo.ai), [github.com/aristoteleo](https://github.com/aristoteleo)).
This notebook only adds a worked example on top of their public tooling and data. See the
[README](../README.md) for license and terms.
